# TimesFM Fine-Tuning Test (TimeCopilot)

This notebook tests **fine-tuning TimesFM using the `timesfm.py` implementation inside TimeCopilot**.

It assumes the repository structure:
`D:/TC/timecopilot/timecopilot/models/foundation/timesfm.py`

Goal:
- Load TimesFM
- Build dataset
- Run `TimesFMFineTuner`
- Verify training works


In [1]:
import sys
import os

# Add TimeCopilot repo to Python path
sys.path.append(r"D:/TC/timecopilot")

print("Path added")

Path added


## Import TimesFM Components
These come from the file you shared:
`timecopilot/models/foundation/timesfm.py`

In [2]:
import pandas as pd
import numpy as np
import torch

from timecopilot.models.foundation.timesfm import (
    TimesFMFineTuner,
    FineTuningConfig,
    TimeSeriesFineTuningDataset
)

from timesfm import TimesFM_2p5_200M_torch

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


## Create Dummy Time Series Dataset
We generate a simple sine wave dataset just to verify fine‑tuning works.

In [3]:
t = np.arange(0, 2000)
y = np.sin(t * 0.02) + np.random.normal(0, 0.05, len(t))

df = pd.DataFrame({"y": y})
df.head()

,y
0,0.001576
1,0.072277
2,-0.018260
3,0.064789
4,0.058262


## Train / Validation Split

In [4]:
train_size = int(len(df) * 0.8)

train_df = df.iloc[:train_size].reset_index(drop=True)
val_df = df.iloc[train_size:].reset_index(drop=True)

print(len(train_df), len(val_df))

1600 400


## Create Fine-Tuning Dataset

In [ ]:
context_length = 32
prediction_length = 16


train_dataset = TimeSeriesFineTuningDataset(
    train_df,
    context_length=context_length,
    prediction_length=prediction_length,
)

val_dataset = TimeSeriesFineTuningDataset(
    val_df,
    context_length=context_length,
    prediction_length=prediction_length,
)

len(train_dataset), len(val_dataset)


(1553, 353)

## Load TimesFM 2.5 PyTorch Model
This is required because **fine-tuning only works with torch-native TimesFM models**.

In [ ]:
from timesfm import TimesFM_2p5_200M_torch
import torch.nn as nn

tfm_wrapper = TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")

backbone = tfm_wrapper.model if hasattr(tfm_wrapper, "model") else tfm_wrapper
assert isinstance(backbone, nn.Module), f"Backbone is not nn.Module: {type(backbone)}"

print("Wrapper type:", type(tfm_wrapper))
print("Backbone type:", type(backbone))


Wrapper type: <class 'timesfm.timesfm_2p5.timesfm_2p5_torch.TimesFM_2p5_200M_torch'>
Backbone type: <class 'timesfm.timesfm_2p5.timesfm_2p5_torch.TimesFM_2p5_200M_torch_module'>


In [7]:
import torch
import torch.nn as nn

class TimesFMForwardAdapter(nn.Module):
    """Adapter so TimeCopilot's fine-tuner (which calls model(context)) can train TimesFM 2.5.

    TimesFM 2.5 torch backbone forward expects patched inputs:
      inputs: (B, n_patches, patch_len)
      masks:  (B, n_patches, patch_len)

    TimeCopilot's fine-tuner provides:
      context: (B, L)

    This adapter:
      1) reshapes (B, L) -> (B, 1, L) treating the whole context as 1 patch
      2) builds an all-ones mask (no padding)
      3) calls backbone(inputs, masks)
      4) unwraps nested (tuple/list) outputs until we get a Tensor
      5) squeezes patch dim if present, and slices/pads to prediction_length -> (B, H)
    """
    def __init__(self, backbone: nn.Module, prediction_length: int, mask_dtype: torch.dtype = torch.bool):
        super().__init__()
        if isinstance(backbone, TimesFMForwardAdapter):
            raise TypeError("Backbone is already a TimesFMForwardAdapter (double-wrapping). Restart kernel or use the original backbone.")
        self.backbone = backbone
        self.prediction_length = int(prediction_length)
        self.mask_dtype = mask_dtype

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() != 2:
            raise ValueError(f"Expected context shape (B, L), got {tuple(x.shape)}")

        # Treat context as a single patch
        x_p = x.unsqueeze(1)  # (B, 1, L)
        masks = torch.ones_like(x_p, dtype=self.mask_dtype, device=x.device)  # (B, 1, L)

        out = self.backbone(x_p, masks)

        # Unwrap nested tuples/lists until we get a Tensor
        while isinstance(out, (tuple, list)):
            out = out[0]

        if not isinstance(out, torch.Tensor):
            raise TypeError(f"Backbone returned unsupported type after unwrapping: {type(out)}")

        # Squeeze singleton patch dimension if present: (B, 1, H) -> (B, H)
        if out.dim() == 3 and out.shape[1] == 1:
            out = out.squeeze(1)

        # Sometimes: (B, H, 1) -> (B, H)
        if out.dim() == 3 and out.shape[-1] == 1:
            out = out.squeeze(-1)

        if out.dim() != 2:
            raise ValueError(f"Expected model output to be 2D (B, H). Got {tuple(out.shape)}")

        # Slice/pad to prediction_length so it matches target shape from TimeSeriesFineTuningDataset
        H = out.shape[1]
        if H > self.prediction_length:
            out = out[:, : self.prediction_length]
        elif H < self.prediction_length:
            pad = torch.zeros((out.shape[0], self.prediction_length - H), device=out.device, dtype=out.dtype)
            out = torch.cat([out, pad], dim=1)

        return out

# Wrap the backbone (DO NOT wrap `model` repeatedly)
model = TimesFMForwardAdapter(backbone, prediction_length=prediction_length)
print("Adapted model type:", type(model))


Adapted model type: <class '__main__.TimesFMForwardAdapter'>


## Configure Fine-Tuning

In [ ]:
config = FineTuningConfig(
    batch_size=8,
    num_epochs=1,
    learning_rate=1e-4,

    
    use_lora=False,
    use_dora=False,
    use_linear_probing=False,

    early_stopping_patience=1,
    use_cosine_schedule=False,
)

config

FineTuningConfig(batch_size=8, num_epochs=1, learning_rate=0.0001, weight_decay=0.01, use_lora=False, use_dora=False, use_linear_probing=False, lora_rank=8, lora_alpha=16.0, lora_dropout=0.1, use_cosine_schedule=False, warmup_steps=0, early_stopping_patience=1, device='cpu', log_every_n_steps=50, checkpoint_dir='./checkpoints')

## Initialize Fine-Tuner

In [9]:
import torch.nn as nn

assert isinstance(model, nn.Module), f"Expected nn.Module, got {type(model)}"
finetuner = TimesFMFineTuner(model, config)
print("FineTuner ready")

2026-03-05 18:29:21,911 - timecopilot.models.foundation.timesfm - INFO - Using Full fine-tuning strategy
INFO:timecopilot.models.foundation.timesfm:Using Full fine-tuning strategy


FineTuner ready


In [ ]:

# Sanity check: ensure we have trainable parameters
trainable = [p for p in model.parameters() if p.requires_grad]
print("Trainable tensors:", len(trainable))
print("Trainable params:", sum(p.numel() for p in trainable))




Trainable tensors: 232
Trainable params: 231289280


## Run Fine-Tuning

In [11]:
result = finetuner.finetune(
    train_dataset=train_dataset,
    val_dataset=val_dataset
)

result

2026-03-05 18:29:23,368 - timecopilot.models.foundation.timesfm - INFO - Starting fine-tuning for 1 epochs
INFO:timecopilot.models.foundation.timesfm:Starting fine-tuning for 1 epochs
2026-03-05 18:29:23,370 - timecopilot.models.foundation.timesfm - INFO - Training samples: 1553
INFO:timecopilot.models.foundation.timesfm:Training samples: 1553
2026-03-05 18:29:23,370 - timecopilot.models.foundation.timesfm - INFO - Validation samples: 353
INFO:timecopilot.models.foundation.timesfm:Validation samples: 353
Training: 100%|██████████| 195/195 [00:12<00:00, 15.76it/s]
2026-03-05 18:29:35,743 - timecopilot.models.foundation.timesfm - INFO - Epoch 1/1 - Train Loss: 0.254488
INFO:timecopilot.models.foundation.timesfm:Epoch 1/1 - Train Loss: 0.254488
Validating: 100%|██████████| 45/45 [00:02<00:00, 20.71it/s]
2026-03-05 18:29:37,921 - timecopilot.models.foundation.timesfm - INFO - Val Loss: 0.071984
INFO:timecopilot.models.foundation.timesfm:Val Loss: 0.071984
2026-03-05 18:29:38,583 - timecopi

{'history': {'train_loss': [0.2544879998247593],
  'val_loss': [0.07198445102613833],
  'learning_rate': [0.0001]},
 'best_val_loss': 0.07198445102613833}